# Milestone 2 - Hugging Face Transformers and Datasets

In [1]:
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import torch
from datasets import load_dataset

DATA_PATH = '/kaggle/input/competitions/smart-mcq-solver-challenge/train.csv'
import os
if not os.path.exists(DATA_PATH):
    DATA_PATH = 'data/train.csv'

# Load with HF datasets (NOT pandas) - this is the requirement for Q1
ds = load_dataset('csv', data_files=DATA_PATH, split='train')
print(f'Loaded: {len(ds)} rows')
print(f'Columns: {ds.column_names}')
print(f'Sample (index 0) prompt: {ds[0]["prompt"][:80]}...')

Generating train split: 0 examples [00:00, ? examples/s]

Loaded: 2000 rows
Columns: ['id', 'prompt', 'A', 'B', 'C', 'D', 'E', 'answer']
Sample (index 0) prompt: Pick the best possible answer: What is Martin Heidegger's view on the relationsh...


In [2]:
def map_at_3(truth, prediction):
    """MAP@3 score for one question. prediction is a list of up to 3 letters."""
    if truth in prediction:
        return 1.0 / (prediction.index(truth) + 1)
    return 0.0

# Sanity check
assert map_at_3('C', ['C', 'A', 'B']) == 1.0
assert map_at_3('B', ['D', 'B', 'E']) == 0.5
print('MAP@3 helper ready.')

MAP@3 helper ready.


In [3]:
# Add the combined_text column via .map()
def add_combined_text(example):
    example['combined_text'] = example['prompt'] + ' ' + example['A']
    return example

ds = ds.map(add_combined_text)

# Pull row at index 51 (zero-indexed)
row_51 = ds[51]
print(f'Index 51 prompt (first 80 chars): {row_51["prompt"][:80]}...')
print(f'Index 51 option A (first 80 chars): {row_51["A"][:80]}...')
print(f'Index 51 combined_text (first 80 chars): {row_51["combined_text"][:80]}...')
print()

q1_answer = len(row_51['combined_text'])
print(f'ANSWER Q1: {q1_answer}')

Map:   0%|          | 0/2000 [00:00<?, ? examples/s]

Index 51 prompt (first 80 chars): Determine the correct option: What is the reason behind the designation of Class...
Index 51 option A (first 80 chars): Class L dwarfs are hotter than M stars and are designated L because L is the rem...
Index 51 combined_text (first 80 chars): Determine the correct option: What is the reason behind the designation of Class...

ANSWER Q1: 614


In [4]:
from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained('bert-base-uncased')

q2_answer = tokenizer.vocab_size
print(f'Tokenizer class: {type(tokenizer).__name__}')
print(f'vocab_size property: {tokenizer.vocab_size}')
print()
print(f'ANSWER Q2: {q2_answer}')

config.json:   0%|          | 0.00/570 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

Tokenizer class: BertTokenizer
vocab_size property: 30522

ANSWER Q2: 30522


In [5]:
sep_token = tokenizer.sep_token
sep_id = tokenizer.sep_token_id

print(f'[SEP] token string: {sep_token!r}')
print(f'[SEP] token ID    : {sep_id}')

# Cross-check by decoding the ID back
print(f'Decode back       : {tokenizer.decode([sep_id])!r}')
print()
print(f'ANSWER Q3: {sep_id}')

[SEP] token string: '[SEP]'
[SEP] token ID    : 102
Decode back       : '[SEP]'

ANSWER Q3: 102


In [6]:
# Pull all prompts and ensure they are plain Python strings
prompts = [str(p) for p in ds['prompt']]

encoded = tokenizer(
    prompts,
    padding='max_length',
    truncation=True,
    max_length=128,
    return_tensors='pt',
)

q4_answer = tuple(encoded['input_ids'].shape)
print(f'Number of prompts tokenized: {len(prompts)}')
print(f'max_length setting: 128')
print(f'input_ids shape: {q4_answer}')
print(f'attention_mask shape: {tuple(encoded["attention_mask"].shape)}')
print()
print(f'ANSWER Q4: {q4_answer}')

Number of prompts tokenized: 2000
max_length setting: 128
input_ids shape: (2000, 128)
attention_mask shape: (2000, 128)

ANSWER Q4: (2000, 128)


In [7]:
HIDDEN_SIZE = 768
NUM_HEADS = 12

q5_answer = HIDDEN_SIZE // NUM_HEADS
print(f'Hidden size: {HIDDEN_SIZE}')
print(f'Number of heads: {NUM_HEADS}')
print(f'Per-head dimension: {HIDDEN_SIZE} / {NUM_HEADS} = {q5_answer}')
print()
print(f'ANSWER Q5: {q5_answer}')

Hidden size: 768
Number of heads: 12
Per-head dimension: 768 / 12 = 64

ANSWER Q5: 64


In [8]:
from transformers import AutoModel

model = AutoModel.from_pretrained('bert-base-uncased')
model.eval()

prompt_0 = ds[0]['prompt']
print(f'Row 0 prompt: {prompt_0[:80]}...')

# Default tokenization - no padding, no truncation, no max_length
inputs = tokenizer(prompt_0, return_tensors='pt')
print(f'Default tokenized input_ids shape: {tuple(inputs["input_ids"].shape)}')
print(f'Number of tokens in this prompt: {inputs["input_ids"].shape[1]}')

# Forward pass (no gradient needed for inference)
with torch.no_grad():
    outputs = model(**inputs)

q6_answer = tuple(outputs.last_hidden_state.shape)
print(f'last_hidden_state shape: {q6_answer}')
print(f'  dim 0 = batch size (1)')
print(f'  dim 1 = sequence length (matches input_ids)')
print(f'  dim 2 = hidden size (768)')
print()
print(f'ANSWER Q6: {q6_answer}')

model.safetensors:   0%|          | 0.00/440M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertModel LOAD REPORT from: bert-base-uncased
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Row 0 prompt: Pick the best possible answer: What is Martin Heidegger's view on the relationsh...
Default tokenized input_ids shape: (1, 31)
Number of tokens in this prompt: 31
last_hidden_state shape: (1, 31, 768)
  dim 0 = batch size (1)
  dim 1 = sequence length (matches input_ids)
  dim 2 = hidden size (768)

ANSWER Q6: (1, 31, 768)


In [9]:
# [CLS] is always at sequence position 0
# last_hidden_state shape: (batch=1, seq_len, hidden=768)
cls_embedding = outputs.last_hidden_state[0, 0, :]
print(f'[CLS] embedding shape: {tuple(cls_embedding.shape)}')

# Pull first 5 values and sum them
first_5 = cls_embedding[:5]
print(f'First 5 values: {first_5.tolist()}')

sum_first_5 = float(first_5.sum())
q7_answer = round(sum_first_5, 4)
print(f'Sum (raw): {sum_first_5}')
print()
print(f'ANSWER Q7: {q7_answer}')

[CLS] embedding shape: (768,)
First 5 values: [-0.46766412258148193, -0.07544441521167755, -0.20190182328224182, -0.007064236328005791, -0.4480222463607788]
Sum (raw): -1.200096845626831

ANSWER Q7: -1.2001


In [10]:
model_att = AutoModel.from_pretrained('bert-base-uncased', output_attentions=True)
model_att.eval()

text = 'Light-ion fusion is a technique.'
inputs_q8 = tokenizer(text, return_tensors='pt')

# Inspect the tokens so we can locate 'fusion' by index
token_ids = inputs_q8['input_ids'][0].tolist()
tokens = tokenizer.convert_ids_to_tokens(token_ids)
print(f'Input text: {text!r}')
print(f'Tokens    : {tokens}')

# Find the index of 'fusion' in the token list
fusion_idx = tokens.index('fusion')
print(f'Index of "fusion": {fusion_idx}')

# Forward pass to get attentions
with torch.no_grad():
    outputs_q8 = model_att(**inputs_q8)

# outputs.attentions is a tuple of length num_layers (12)
# Each element has shape: (batch, num_heads, seq_len, seq_len)
attentions = outputs_q8.attentions
print(f'Number of layers: {len(attentions)}')
print(f'Per-layer attention shape: {tuple(attentions[-1].shape)}')

# Last layer, head 0 -> drop batch dim -> (seq_len, seq_len)
last_layer_head0 = attentions[-1][0, 0]
print(f'Last layer, head 0 attention matrix shape: {tuple(last_layer_head0.shape)}')

# Attention that token 0 ([CLS]) pays to token 'fusion'
attn_weight = float(last_layer_head0[0, fusion_idx])
q8_answer = round(attn_weight, 4)
print(f'Attention weight [CLS] (index 0) -> "fusion" (index {fusion_idx}): {attn_weight}')
print()
print(f'ANSWER Q8: {q8_answer}')

# Free memory before Q9
del model_att
import gc; gc.collect()


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertModel LOAD REPORT from: bert-base-uncased
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Input text: 'Light-ion fusion is a technique.'
Tokens    : ['[CLS]', 'light', '-', 'ion', 'fusion', 'is', 'a', 'technique', '.', '[SEP]']
Index of "fusion": 4
Number of layers: 12
Per-layer attention shape: (1, 12, 10, 10)
Last layer, head 0 attention matrix shape: (10, 10)
Attention weight [CLS] (index 0) -> "fusion" (index 4): 0.10247313976287842

ANSWER Q8: 0.1025


1124

In [11]:
from sentence_transformers import SentenceTransformer, util

st_model = SentenceTransformer('sentence-transformers/all-MiniLM-L6-v2')

prompt_0 = ds[0]['prompt']
option_b_0 = ds[0]['B']
print(f'Row 0 prompt  (first 80): {prompt_0[:80]}...')
print(f'Row 0 option B (first 80): {option_b_0[:80]}...')

# Encode each text -> 384-dim embedding tensors
emb_prompt = st_model.encode(prompt_0, convert_to_tensor=True)
emb_option_b = st_model.encode(option_b_0, convert_to_tensor=True)
print(f'Embedding shape: {tuple(emb_prompt.shape)}')

# Cosine similarity via sentence_transformers.util.cos_sim (as required)
sim_tensor = util.cos_sim(emb_prompt, emb_option_b)
sim_value = float(sim_tensor[0][0])
q9_answer = round(sim_value, 4)
print(f'cos_sim output: {sim_tensor}')
print(f'Similarity: {sim_value}')
print()
print(f'ANSWER Q9: {q9_answer}')

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

...

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Row 0 prompt  (first 80): Pick the best possible answer: What is Martin Heidegger's view on the relationsh...
Row 0 option B (first 80): Martin Heidegger believes that humans do not exist inside time, but that they ar...
Embedding shape: (384,)
cos_sim output: tensor([[0.7658]], device='cuda:0')
Similarity: 0.7658098340034485

ANSWER Q9: 0.7658


In [12]:
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity as sklearn_cos_sim

# Reload as pandas for the pipeline work (datasets is awkward for row iteration)
df = pd.read_csv(DATA_PATH)
print(f'Loaded {len(df)} rows for pipeline evaluation')

Loaded 2000 rows for pipeline evaluation


In [13]:
# ------------------------------------------------------------------
# Pipeline 1: TF-IDF cosine similarity (same as Milestone 1)
# ------------------------------------------------------------------
def combine_row_text(row):
    parts = [str(row['prompt'])]
    for opt in 'ABCDE':
        parts.append(str(row[opt]))
    return ' '.join(parts)

combined_docs = df.apply(combine_row_text, axis=1).tolist()
tfidf = TfidfVectorizer(stop_words='english')
tfidf.fit(combined_docs)

# Pre-transform all prompts once (saves 5x recompute)
prompt_vectors = tfidf.transform(df['prompt'].astype(str).tolist())

tfidf_top3_list = []
for i in range(len(df)):
    row = df.iloc[i]
    p_vec = prompt_vectors[i]
    opt_texts = [str(row[opt]) for opt in 'ABCDE']
    opt_vecs = tfidf.transform(opt_texts)
    sims = sklearn_cos_sim(p_vec, opt_vecs)[0]
    sorted_idx = np.argsort(sims)[::-1]
    top3 = ['ABCDE'[j] for j in sorted_idx[:3]]
    tfidf_top3_list.append(top3)

tfidf_map3 = float(np.mean([
    map_at_3(df.iloc[i]['answer'], tfidf_top3_list[i])
    for i in range(len(df))
]))
print(f'TF-IDF pipeline MAP@3: {tfidf_map3:.4f}')

TF-IDF pipeline MAP@3: 0.2552


In [14]:
# ------------------------------------------------------------------
# Pipeline 2: MiniLM embeddings + cosine similarity
# ------------------------------------------------------------------
# Batch-encode all prompts and all 5 option columns at once
prompts_list = df['prompt'].astype(str).tolist()
option_lists = {opt: df[opt].astype(str).tolist() for opt in 'ABCDE'}

print('Encoding all prompts (1 batch)...')
prompt_embs = st_model.encode(prompts_list, convert_to_tensor=True,
                              batch_size=64, show_progress_bar=False)
print(f'Prompt embeddings shape: {tuple(prompt_embs.shape)}')

option_embs = {}
for opt in 'ABCDE':
    print(f'Encoding all option {opt} (1 batch)...')
    option_embs[opt] = st_model.encode(option_lists[opt], convert_to_tensor=True,
                                       batch_size=64, show_progress_bar=False)

# For each row, compute cosine similarity between prompt embedding and 5 option embeddings
minilm_top3_list = []
minilm_scores = []
for i in range(len(df)):
    p_emb = prompt_embs[i]
    sims = [float(util.cos_sim(p_emb, option_embs[opt][i])[0][0]) for opt in 'ABCDE']
    sorted_idx = np.argsort(sims)[::-1]
    top3 = ['ABCDE'[j] for j in sorted_idx[:3]]
    minilm_top3_list.append(top3)
    minilm_scores.append(map_at_3(df.iloc[i]['answer'], top3))

minilm_map3 = float(np.mean(minilm_scores))
q10_map3 = round(minilm_map3, 4)
print(f'MiniLM pipeline MAP@3: {minilm_map3}')
print()
print(f'ANSWER Q10 (a) - MiniLM MAP@3: {q10_map3}')


Encoding all prompts (1 batch)...
Prompt embeddings shape: (2000, 384)
Encoding all option A (1 batch)...
Encoding all option B (1 batch)...
Encoding all option C (1 batch)...
Encoding all option D (1 batch)...
Encoding all option E (1 batch)...
MiniLM pipeline MAP@3: 0.4230833333333333

ANSWER Q10 (a) - MiniLM MAP@3: 0.4231


In [15]:
# ------------------------------------------------------------------
# Count questions where TF-IDF missed the truth but MiniLM caught it
# ------------------------------------------------------------------
count = 0
for i in range(len(df)):
    truth = df.iloc[i]['answer']
    in_tfidf = truth in tfidf_top3_list[i]
    in_minilm = truth in minilm_top3_list[i]
    if (not in_tfidf) and in_minilm:
        count += 1

q10_count = count
print(f'Total questions: {len(df)}')
print(f'Cases where TF-IDF top-3 missed AND MiniLM top-3 caught: {count}')
print()
print(f'ANSWER Q10 (b) - Count: {q10_count}')

# Quick sanity numbers
both_hit = sum(1 for i in range(len(df))
               if df.iloc[i]['answer'] in tfidf_top3_list[i]
               and df.iloc[i]['answer'] in minilm_top3_list[i])
tfidf_only = sum(1 for i in range(len(df))
                 if df.iloc[i]['answer'] in tfidf_top3_list[i]
                 and df.iloc[i]['answer'] not in minilm_top3_list[i])
minilm_only = q10_count
neither = len(df) - both_hit - tfidf_only - minilm_only
print()
print('Overlap breakdown:')
print(f'  both hit        : {both_hit}')
print(f'  TF-IDF only hit : {tfidf_only}')
print(f'  MiniLM only hit : {minilm_only}')
print(f'  neither hit     : {neither}')

# Free the MiniLM model before Q11 (which loads bart-large-mnli)
del st_model
gc.collect()


Total questions: 2000
Cases where TF-IDF top-3 missed AND MiniLM top-3 caught: 613

ANSWER Q10 (b) - Count: 613

Overlap breakdown:
  both hit        : 681
  TF-IDF only hit : 249
  MiniLM only hit : 613
  neither hit     : 457


2104

In [16]:
from transformers import pipeline as hf_pipeline

# Defaults to facebook/bart-large-mnli
zs = hf_pipeline('zero-shot-classification')

prompt_1 = df.iloc[1]['prompt']
candidate_labels = [str(df.iloc[1]['A']), str(df.iloc[1]['B']), str(df.iloc[1]['C'])]
print(f'Index 1 prompt: {prompt_1[:90]}...')
print(f'Candidate labels (first 60 chars each):')
for c in candidate_labels:
    print(f'  - {c[:60]}...')
print()

result = zs(prompt_1, candidate_labels=candidate_labels)
print(f'Top label (first 60 chars): {str(result["labels"][0])[:60]}...')
print(f'Top probability: {result["scores"][0]}')
print(f'All scores: {result["scores"]}')
print(f'Sum of 3 scores (softmax, should be 1.0): {sum(result["scores"])}')

q11_top_prob = round(result['scores'][0], 4)
q11_sum = float(sum(result['scores']))
print()
print(f'ANSWER Q11: {q11_top_prob}')

No model was supplied, defaulted to facebook/bart-large-mnli and revision d7645e1.
Using a pipeline without specifying a model name and revision in production is not recommended.


config.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/1.63G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/515 [00:00<?, ?it/s]

tokenizer_config.json:   0%|          | 0.00/26.0 [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

Index 1 prompt: What is accelerator-based light-ion fusion?...
Candidate labels (first 60 chars each):
  - Accelerator-based light-ion fusion is a technique that uses ...
  - Accelerator-based light-ion fusion is a technique that uses ...
  - Accelerator-based light-ion fusion is a technique that uses ...

Top label (first 60 chars): Accelerator-based light-ion fusion is a technique that uses ...
Top probability: 0.4574522376060486
All scores: [0.4574522376060486, 0.2750644385814667, 0.26748329401016235]
Sum of 3 scores (softmax, should be 1.0): 0.9999999701976776

ANSWER Q11: 0.4575


In [17]:
result_ml = zs(prompt_1, candidate_labels=candidate_labels, multi_label=True)
print(f'Top label (first 60 chars): {str(result_ml["labels"][0])[:60]}...')
print(f'Top probability: {result_ml["scores"][0]}')
print(f'All scores: {result_ml["scores"]}')
print(f'Sum of 3 scores (sigmoid, usually > 1.0): {sum(result_ml["scores"])}')

q12_sum = float(sum(result_ml['scores']))
diff = abs(q11_sum - q12_sum)
q12_answer = round(diff, 4)
print()
print(f'Q11 sum (softmax): {q11_sum}')
print(f'Q12 sum (sigmoid): {q12_sum}')
print(f'Absolute difference: {diff}')
print()
print(f'ANSWER Q12: {q12_answer}')

# Free memory before Q13
del zs
gc.collect()

Top label (first 60 chars): Accelerator-based light-ion fusion is a technique that uses ...
Top probability: 0.00046926175127737224
All scores: [0.00046926175127737224, 2.063553074549418e-05, 1.9700279153767042e-05]
Sum of 3 scores (sigmoid, usually > 1.0): 0.0005095975611766335

Q11 sum (softmax): 0.9999999701976776
Q12 sum (sigmoid): 0.0005095975611766335
Absolute difference: 0.999490372636501

ANSWER Q12: 0.9995


16

In [18]:
from transformers import pipeline

t5 = pipeline(
    "text-generation",
    model="google/flan-t5-small"
)

prompt_0 = df.iloc[0]['prompt']
A_0 = df.iloc[0]['A']
B_0 = df.iloc[0]['B']

# Construct the exact input string as specified
input_str = (
    f'Question: {prompt_0}. '
    f'Is the correct answer A: {A_0} or B: {B_0}? '
    f'Answer with just the letter A or B.'
)
print(f'Input string length: {len(input_str)}')
print(f'Input string (first 200 chars): {input_str[:200]}...')
print()

result = t5(input_str, max_new_tokens=5)
# result is a list of dicts: [{'generated_text': '...'}]
q13_answer = result[0]['generated_text']
print(f'Raw pipeline result: {result}')
print()
print(f'ANSWER Q13: {q13_answer!r}')


config.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/308M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/190 [00:00<?, ?it/s]

The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning


generation_config.json:   0%|          | 0.00/147 [00:00<?, ?B/s]

...

special_tokens_map.json: 0.00B [00:00, ?B/s]

The model 'T5ForConditionalGeneration' is not supported for text-generation. Supported models are ['PeftModelForCausalLM', 'AfmoeForCausalLM', 'ApertusForCausalLM', 'ArceeForCausalLM', 'AriaTextForCausalLM', 'BambaForCausalLM', 'BartForCausalLM', 'BertLMHeadModel', 'BertGenerationDecoder', 'BigBirdForCausalLM', 'BigBirdPegasusForCausalLM', 'BioGptForCausalLM', 'BitNetForCausalLM', 'BlenderbotForCausalLM', 'BlenderbotSmallForCausalLM', 'BloomForCausalLM', 'BltForCausalLM', 'CamembertForCausalLM', 'LlamaForCausalLM', 'CodeGenForCausalLM', 'CohereForCausalLM', 'Cohere2ForCausalLM', 'CpmAntForCausalLM', 'CTRLLMHeadModel', 'CwmForCausalLM', 'Data2VecTextForCausalLM', 'DbrxForCausalLM', 'DeepseekV2ForCausalLM', 'DeepseekV3ForCausalLM', 'DiffLlamaForCausalLM', 'DogeForCausalLM', 'Dots1ForCausalLM', 'ElectraForCausalLM', 'Emu3ForCausalLM', 'ErnieForCausalLM', 'Ernie4_5ForCausalLM', 'Ernie4_5_MoeForCausalLM', 'Exaone4ForCausalLM', 'FalconForCausalLM', 'FalconH1ForCausalLM', 'FalconMambaForCausa

Input string length: 788
Input string (first 200 chars): Question: Pick the best possible answer: What is Martin Heidegger's view on the relationship between time and human existence? among the listed options.. Is the correct answer A: Martin Heidegger beli...



Both `max_new_tokens` (=5) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Raw pipeline result: [{'generated_text': "Question: Pick the best possible answer: What is Martin Heidegger's view on the relationship between time and human existence? among the listed options.. Is the correct answer A: Martin Heidegger believes that humans exist within a time continuum that is infinite and does not have a defined beginning or end. The relationship to the past involves acknowledging it as a historical era, and the relationship to the future involves creating a world that will endure beyond one's own time. or B: Martin Heidegger believes that humans do not exist inside time, but that they are time. The relationship to the past is a present awareness of having been, and the relationship to the future involves anticipating a potential possibility, task, or engagement.? Answer with just the letter A or B."}]

ANSWER Q13: "Question: Pick the best possible answer: What is Martin Heidegger's view on the relationship between time and human existence? among the listed options.